# Part 1 — OLS, Hat Matrix, Ridge Regression & VIF

**Group 12 — Data Fitting and OLS Methods**

Notebook này kiểm chứng 4 hàm cài đặt:
- `ols_fit` — OLS qua Economic SVD (`utils/svd_solver.py → utils/decomposition.py`)
- `hat_matrix` — Ma trận chiếu H = U_r Uᵣᵀ qua SVD
- `ridge_fit` — Ridge Regression qua Gauss-Jordan inverse
- `vif` — Variance Inflation Factor qua hồi quy phụ OLS

In [ ]:
import sys, os
# Them thu muc goc du an vao sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from part1.ols_implementation import ols_fit, hat_matrix, vif
from part1.ridge_lasso import ridge_fit
from utils.decomposition import svd_decomp   # ham SVD

rng = np.random.default_rng(42)
print('Import thanh cong!')
print('svd_decomp den tu utils/decomposition.py')

---
## 1. OLS Fit — `ols_fit(X, y)`

### Lý thuyết
Ước lượng OLS tối thiểu hóa $\|y - X\beta\|^2$, cho nghiệm:
$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

### Cài đặt
Dùng Economic SVD: $X = U\Sigma V^\top$
$$\hat{\beta} = V\Sigma^{-1}U^\top y$$

Luồng gọi: `ols_fit` → `svd_solve` (utils/svd_solver.py) → `economic_svd` → **`svd_decomp`** (utils/decomposition.py)

In [ ]:
# Sinh du lieu gia lap
n, p = 100, 3
X_raw = rng.standard_normal((n, p))
X = np.hstack([np.ones((n, 1)), X_raw])   # them cot he so chan
true_beta = np.array([2.0, 1.5, -1.0, 0.5])
y = X @ true_beta + rng.standard_normal(n) * 1.5

# OLS
beta_hat, _ = ols_fit(X, y)

# Kiem chung bang numpy.linalg.lstsq
beta_ref = np.linalg.lstsq(X, y, rcond=None)[0]

print('=== OLS Fit Verification ===')
print(f'beta_hat (our SVD) : {beta_hat}')
print(f'beta_ref (lstsq)   : {beta_ref}')
print(f'Max absolute error : {np.max(np.abs(beta_hat - beta_ref)):.2e}')
assert np.allclose(beta_hat, beta_ref, atol=1e-8), 'FAIL: ols_fit khong khop voi lstsq!'
print('PASS: ols_fit khop voi numpy.linalg.lstsq den do chinh xac may!')

---
## 2. Hat Matrix — `hat_matrix(X)`

### Lý thuyết
$$H = X(X^\top X)^{-1}X^\top, \quad \hat{y} = Hy$$

### Cài đặt qua SVD
$$H = U_r U_r^\top \quad \text{(với } U_r \text{ là các cột của } U \text{ ứng với } \sigma_i > \varepsilon\text{)}$$

Tính chất cần kiểm chứng: lũy đẳng ($H^2=H$), đối xứng ($H^\top=H$), $\text{tr}(H)=p+1$.

In [ ]:
H = hat_matrix(X)

H2_err  = np.max(np.abs(H @ H - H))
sym_err = np.max(np.abs(H - H.T))
tr_val  = np.trace(H)

print('=== Hat Matrix Properties ===')
print(f'Shape                    : {H.shape}')
print(f'Idempotent |H^2-H| max   : {H2_err:.2e}  ({"PASS" if H2_err < 1e-8 else "FAIL"})')
print(f'Symmetric  |H-H^T| max   : {sym_err:.2e}  ({"PASS" if sym_err < 1e-8 else "FAIL"})')
print(f'tr(H) = {tr_val:.6f}  (expected {p+1})  ({"PASS" if abs(tr_val-(p+1)) < 1e-8 else "FAIL"})')

# Kiem chung bang tinh truc tiep
H_ref = X @ np.linalg.solve(X.T @ X, X.T)
print(f'Max diff vs direct H_ref : {np.max(np.abs(H - H_ref)):.2e}')

---
## 3. Ridge Regression — `ridge_fit(X, y, lam)`

### Lý thuyết
$$\hat{\beta}_{\text{ridge}} = (X^\top X + \lambda I)^{-1} X^\top y$$

### Cài đặt
Dùng Gauss-Jordan inverse (`utils/inverse.py`). Ma trận $(X^\top X + \lambda I)$ luôn khả nghịch với $\lambda > 0$.

In [ ]:
# Du lieu co da cong tuyen
X_mc = np.hstack([np.ones((n,1)), X_raw[:,0:1], X_raw[:,1:2],
                  0.95*X_raw[:,0:1] + 0.05*rng.standard_normal((n,1))])
y_mc = X_mc @ np.array([1,2,-1,0.5]) + rng.standard_normal(n)*1.5

lambdas = np.logspace(-3, 2, 50)
betas_ridge = [ridge_fit(X_mc, y_mc, lam) for lam in lambdas]

# Ridge Trace
plt.figure(figsize=(8, 4))
for j in range(X_mc.shape[1]):
    plt.plot(np.log10(lambdas), [b[j] for b in betas_ridge], label=f'beta[{j}]')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('log10(lambda)')
plt.ylabel('Coefficient value')
plt.title('Ridge Trace (he so tieu dan ve 0 khi lambda tang)')
plt.legend()
plt.tight_layout()
plt.show()

# Kiem chung tai mot lambda cu the
lam_test = 1.0
b_ours = ridge_fit(X_mc, y_mc, lam_test)
b_ref  = np.linalg.solve(X_mc.T@X_mc + lam_test*np.eye(X_mc.shape[1]), X_mc.T@y_mc)
print(f'=== Ridge Fit Verification (lambda={lam_test}) ===')
print(f'Max error vs numpy.linalg.solve: {np.max(np.abs(b_ours - b_ref)):.2e}')
assert np.allclose(b_ours, b_ref, atol=1e-8), 'FAIL'
print('PASS')

---
## 4. VIF — `vif(X)`

### Lý thuyết
$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

$R_j^2$ là hệ số xác định khi hồi quy $x_j$ lên các biến còn lại. Cài đặt: gọi `ols_fit` cho từng biến, tính $R^2$ bằng vòng lặp.

In [ ]:
print('=== Scenario 1: Bien doc lap (VIF ~ 1) ===')
X_ind = rng.standard_normal((n, 3))
vif_df = vif(X_ind)
print(vif_df.to_string(index=False))
assert vif_df['VIF_Score'].max() < 5, 'FAIL: VIF qua cao cho du lieu doc lap'
print('PASS: VIF gan 1 khi cac bien doc lap\n')

print('=== Scenario 2: Da cong tuyen manh (VIF >> 10) ===')
X_col = np.column_stack([
    X_ind[:,0],
    X_ind[:,1],
    X_ind[:,0]*0.99 + 0.01*rng.standard_normal(n)
])
vif_df2 = vif(X_col)
print(vif_df2.to_string(index=False))
assert vif_df2['VIF_Score'].max() > 10, 'FAIL: VIF phai cao khi co da cong tuyen'
print('PASS: VIF rat cao khi co bien gan tuong quan tuyen tinh')

---
## Tổng kết

| Hàm | Phương pháp | Kiểm chứng | Kết quả |
|---|---|---|---|
| `ols_fit` | Economic SVD (`svd_decomp`) | vs `numpy.linalg.lstsq` | max err < 1e-8 |
| `hat_matrix` | H = UᵣUᵣᵀ từ SVD | H²=H, H=Hᵀ, tr(H)=p+1 | max err < 1e-16 |
| `ridge_fit` | Gauss-Jordan inverse | vs `numpy.linalg.solve` | max err < 1e-8 |
| `vif` | Hồi quy phụ OLS + R² | VIF~1 (độc lập), VIF>>10 (cộng tuyến) | PASS |

**Tất cả SVD đều dùng `svd_decomp` từ `utils/decomposition.py`. `np.linalg.svd` không được sử dụng trong bất kỳ hàm nào.**